In [1]:
q1 = "I just discovered the course. Can I still join it?"
q2 = "I just found out about the program. Can I still enroll?"

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
print("Model loaded")

/workspaces/LLM-zoomcamp-02-vector-search/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:01<00:00, 53.79it/s]


Model loaded


In [3]:
v1 = model.encode(q1)

In [4]:
v1.shape

(384,)

In [5]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [6]:
v1.dot(dv)

np.float32(0.38779145)

In [7]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [8]:
v2.dot(dv)

np.float32(0.019730574)

In [9]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-29 16:44:09--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py.3’

ingest.py.3         100%[===================>]     888  --.-KB/s    in 0s      

2026-06-29 16:44:09 (17.0 MB/s) - ‘ingest.py.3’ saved [888/888]



In [10]:
from ingest import load_faq_data

documents = load_faq_data()

In [11]:
documents[0]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.",
 'doc_id': '9e508f2212'}

In [12]:
texts = []

for doc in documents: 
    text = doc['question'] + " " + doc['answer']
    texts.append(text)

In [13]:
texts[10]

'Course: How many hours per week am I expected to spend on this course? It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'

In [14]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

100%|██████████| 27/27 [01:06<00:00,  2.47s/it]


1350

In [15]:
vectors[10].shape

(384,)

In [16]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [17]:
scores[:10]

[np.float32(0.4204169),
 np.float32(0.2730465),
 np.float32(0.5959129),
 np.float32(0.42628884),
 np.float32(0.3280712),
 np.float32(0.4401735),
 np.float32(0.29916102),
 np.float32(0.562418),
 np.float32(0.59272504),
 np.float32(0.27858168)]

In [18]:
import numpy as np
X = np.array(vectors)

In [20]:
X

array([[-0.02670618, -0.12245757,  0.01594413, ..., -0.00230654,
        -0.11218394, -0.02365559],
       [-0.01099552, -0.11074744, -0.02536942, ...,  0.09022228,
        -0.02697371,  0.01975672],
       [-0.08896548, -0.06128178,  0.00775603, ...,  0.0405971 ,
         0.00479277, -0.02745943],
       ...,
       [-0.03652925,  0.01415426, -0.06838644, ...,  0.04316786,
         0.08105537, -0.02148626],
       [-0.13091588, -0.06990605, -0.0093188 , ..., -0.00044342,
        -0.0128573 ,  0.01426918],
       [-0.07984784,  0.01926981,  0.02544978, ..., -0.03368027,
        -0.01884026,  0.05837054]], shape=(1350, 384), dtype=float32)

In [21]:
scores = X.dot(v1)

In [22]:
scores

array([ 0.4204169 ,  0.2730465 ,  0.59591293, ..., -0.04286757,
        0.08712129, -0.02408037], shape=(1350,), dtype=float32)

In [23]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(538), np.float32(0.83650464))

In [24]:
documents[idx]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [30]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]

In [31]:
scores[top5]

array([0.83650464, 0.69035655, 0.6042597 , 0.59591293, 0.59272504],
      dtype=float32)

In [32]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.83650464
{'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', 'doc_id': '74eb249bbf'}

0.69035655
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.', 'doc_id': '41aabbd7c5'}

0.6042597
{'course': 'mlops-zoomcamp', 'section': 'G

In [33]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [35]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [36]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [39]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-29 17:40:15--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-29 17:40:16 (25.6 MB/s) - ‘rag_helper.py’ saved [2134/2134]

--2026-06-29 17:40:16--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK

In [43]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [44]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [45]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [46]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [47]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [48]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [49]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join even if the program has already begun. If you want to receive a certificate, make sure you submit your project while submissions are still open.'

In [50]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [51]:
vs_index.fit(vectors, documents)

In [52]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [53]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [54]:
vs_index.close()